<a href="https://colab.research.google.com/github/HudaSaffo/fashion-recommendation-system/blob/week5-rag-layer/RAG_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5 — Rules-Based RAG for Customer Query Understanding

This notebook turns a customer's fashion request into structured intent and retrieves matching product candidates from the refined catalog.

The pipeline:
1. Retrieves relevant style rules, product information, and existing outfits.
2. Uses rules to extract the customer's occasion, style, preferences to avoid, season, categories, colors, and price limit.
3. Filters and ranks real catalog products for each requested category.

Products come from `catalog_refined.parquet`. Existing outfits are built from products sharing a `set_id`; `item_id` identifies each product. Recommendations use `refined_category` and `refined_color`.

The output is **ranked product candidates**, not a complete styled outfit.

### Project file paths

This cell records where the refined product catalog and segmentation manifest are stored in Google Drive. Later cells use these paths to load the real project data.

In [2]:
from pathlib import Path

CATALOG_PATH = Path('/content/drive/MyDrive/catalog_refined.parquet')
SEGMENTATION_MANIFEST_PATH = Path(
    '/content/drive/MyDrive/polyvore_segmented_cache/segmentation_manifest.csv'
)

### Install dependencies

This cell installs the Python packages used by the notebook. The rules-based pipeline does not need an OpenAI API key.

In [3]:
%pip install -q pandas pyarrow faiss-cpu open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


### Structured customer intent

This cell defines the fields used to represent a customer request: occasion, style, things to avoid, season, needed categories, color palette, and price constraints. It also checks for issues such as an unsupported category or a conflict between a desired and an avoided style. Bags are a separate category; hats and eyeglasses are accessories.

In [4]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from typing import Any

SUPPORTED_CATEGORIES = {"tops", "bottoms", "dresses", "outerwear", "shoes", "bags", "accessories"}


def _strings(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        value = [value]
    if not isinstance(value, list):
        raise ValueError("must be a string list")
    return list(dict.fromkeys(str(x).strip().lower() for x in value if str(x).strip()))


@dataclass
class PriceConstraints:
    minimum: float | None = None
    maximum: float | None = None
    currency: str | None = None

    @classmethod
    def from_dict(cls, value: Any) -> "PriceConstraints | None":
        if value in (None, {}):
            return None
        if not isinstance(value, dict):
            raise ValueError("price_constraints must be an object or null")
        minimum = float(value["minimum"]) if value.get("minimum") is not None else None
        maximum = float(value["maximum"]) if value.get("maximum") is not None else None
        if minimum is not None and minimum < 0 or maximum is not None and maximum < 0:
            raise ValueError("prices cannot be negative")
        if minimum is not None and maximum is not None and minimum > maximum:
            raise ValueError("minimum price cannot exceed maximum")
        currency = str(value["currency"]).upper() if value.get("currency") else None
        return cls(minimum, maximum, currency)


@dataclass
class ValidationIssue:
    code: str
    message: str
    severity: str = "warning"


@dataclass
class CustomerIntent:
    occasion: str | None = None
    style: list[str] = field(default_factory=list)
    avoid: list[str] = field(default_factory=list)
    season: str | None = None
    categories_needed: list[str] = field(default_factory=list)
    color_palette: list[str] = field(default_factory=list)
    price_constraints: PriceConstraints | None = None

    @classmethod
    def from_dict(cls, data: Any, strict_categories: bool = False) -> "CustomerIntent":
        if not isinstance(data, dict):
            raise ValueError("intent must be a JSON object")
        categories = _strings(data.get("categories_needed"))
        unsupported = set(categories) - SUPPORTED_CATEGORIES
        if strict_categories and unsupported:
            raise ValueError(f"unsupported categories: {', '.join(sorted(unsupported))}")
        return cls(
            occasion=str(data["occasion"]).strip().lower() if data.get("occasion") else None,
            style=_strings(data.get("style")), avoid=_strings(data.get("avoid")),
            season=str(data["season"]).strip().lower() if data.get("season") else None,
            categories_needed=categories, color_palette=_strings(data.get("color_palette")),
            price_constraints=PriceConstraints.from_dict(data.get("price_constraints")),
        )

    def validate(self) -> list[ValidationIssue]:
        issues = []
        unsupported = sorted(set(self.categories_needed) - SUPPORTED_CATEGORIES)
        if unsupported:
            issues.append(ValidationIssue("unsupported_category", f"Unsupported: {', '.join(unsupported)}"))
        conflicts = sorted(set(self.style) & set(self.avoid))
        if conflicts:
            issues.append(ValidationIssue("preference_conflict", f"Desired and avoided: {', '.join(conflicts)}"))
        if self.season and self.season not in {"spring", "summer", "autumn", "fall", "winter", "all-season"}:
            issues.append(ValidationIssue("unknown_season", f"Unknown season: {self.season}"))
        return issues

    def supported_categories(self) -> list[str]:
        return [x for x in self.categories_needed if x in SUPPORTED_CATEGORIES]

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


INTENT_JSON_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "occasion": {"type": ["string", "null"]},
        "style": {"type": "array", "items": {"type": "string"}},
        "avoid": {"type": "array", "items": {"type": "string"}},
        "season": {"type": ["string", "null"]},
        "categories_needed": {"type": "array", "items": {"type": "string"}},
        "color_palette": {"type": "array", "items": {"type": "string"}},
        "price_constraints": {"anyOf": [{"type": "null"}, {"type": "object", "additionalProperties": False,
            "properties": {"minimum": {"type": ["number", "null"]}, "maximum": {"type": ["number", "null"]}, "currency": {"type": ["string", "null"]}},
            "required": ["minimum", "maximum", "currency"]}]},
    },
    "required": ["occasion", "style", "avoid", "season", "categories_needed", "color_palette", "price_constraints"],
}

### Normalize catalog products

This cell converts a row from our known catalog into the format used by the recommendation pipeline. It uses `item_id` as the product ID, `refined_category` as the category, `refined_color` as the color, and `real_price` as the price.

In [5]:
from __future__ import annotations

import math
from typing import Any

def _clean(value: Any) -> str:
    return "" if value is None or isinstance(value, float) and math.isnan(value) else " ".join(str(value).strip().split())


def _list(value: Any) -> list[str]:
    if isinstance(value, list): return [str(x).strip().lower() for x in value]
    return [x.strip().lower() for x in _clean(value).replace("|", ",").split(",") if x.strip()]


def normalize_product(row: dict[str, Any]) -> dict[str, Any]:
    price = row.get("real_price")
    try: price = float(price) if _clean(price) else None
    except (TypeError, ValueError): price = None
    if price is not None and math.isnan(price): price = None
    return {
        "product_id": _clean(row["item_id"]),
        "title": _clean(row.get("title")).lower(),
        "description": _clean(row.get("description")).lower(),
        "category": _clean(row["refined_category"]).lower(),
        "color": _clean(row["refined_color"]).lower(),
        "material": _list(row.get("material")),
        "season": _list(row.get("season")),
        "style": _list(row.get("style")),
        "occasion": _list(row.get("occasion")),
        "price": price,
        "outfit_ids": [],
    }

### Retrieve fashion knowledge

This cell builds searchable knowledge from style rules, category and color rules, products, and existing outfits. For each customer query, it retrieves relevant context. The current notebook uses lexical (word-based) retrieval unless an embedding backend is configured.

In [6]:
from __future__ import annotations

import math, re
from collections import Counter
from dataclasses import dataclass
from typing import Any, Protocol

TOKEN = re.compile(r"[a-z0-9]+")

STYLE_GUIDES = [
    ("style", "Smart casual balances polished pieces with relaxed materials; it suits elegant events that are not too formal."),
    ("season", "Summer outfits favor breathable cotton or linen, light colors, and lighter layers."),
    ("occasion", "A dinner outfit can combine a refined top, bottom, shoes, and an optional accessory."),
    ("style", "Formal styling favors tailoring and dress shoes; casual styling favors denim, knits, and sneakers."),
]
CATEGORY_RULES = [
    ("compatibility", "A separates outfit uses tops, bottoms, shoes, and optionally bags, accessories, or outerwear."),
    ("compatibility", "A dress from dresses replaces tops and bottoms; combine it with shoes and optionally bags, accessories, or outerwear."),
]
COLOR_RULES = [
    ("color", "Neutral palettes include black, white, cream, beige, grey, navy, and brown."),
    ("color", "Light warm palettes include cream, beige, ivory, tan, camel, peach, coral, and gold."),
    ("color", "Analogous or neutral colors are easy to combine; use an accessory for a controlled accent."),
]


@dataclass
class KnowledgeChunk:
    chunk_id: str
    source: str
    text: str
    metadata: dict[str, Any]
    score: float = 0.0

    def to_dict(self): return {"chunk_id": self.chunk_id, "source": self.source, "text": self.text, "score": round(self.score, 4)}


class EmbeddingBackend(Protocol):
    def rank(self, query: str, chunks: list[KnowledgeChunk], top_k: int) -> list[KnowledgeChunk]: ...


class KnowledgeRetriever:
    def __init__(self, products, outfits=None, embedding_backend: EmbeddingBackend | None = None):
        self.embedding_backend = embedding_backend
        self.last_diagnostics = {"retrieval_mode": "lexical", "embedding_fallback_reason": None}
        self.chunks = []
        for i, (source, text) in enumerate(STYLE_GUIDES + CATEGORY_RULES + COLOR_RULES):
            self.chunks.append(KnowledgeChunk(f"rule:{i}", source, text, {}))
        for p in products:
            text = " ".join([p["title"], p["description"], p["category"], p["color"], *p["material"], *p["season"], *p["style"], *p["occasion"]])
            self.chunks.append(KnowledgeChunk(f"product:{p['product_id']}", "product", text, {"product_id": p["product_id"]}))
        for outfit in outfits or []:
            text = " ".join(str(outfit.get(k, "")) for k in ("name", "description", "style", "occasion", "season", "colors"))
            self.chunks.append(KnowledgeChunk(f"outfit:{outfit.get('outfit_id')}", "outfit", text, dict(outfit)))

    def retrieve(self, query: str, top_k: int = 8) -> list[KnowledgeChunk]:
        if self.embedding_backend:
            try:
                ranked = self.embedding_backend.rank(query, self.chunks, top_k)
                self.last_diagnostics = {"retrieval_mode": "week4_semantic", "embedding_fallback_reason": None}
                return ranked
            except Exception as exc:
                self.last_diagnostics = {
                    "retrieval_mode": "lexical",
                    "embedding_fallback_reason": type(exc).__name__,
                }
        else:
            self.last_diagnostics = {"retrieval_mode": "lexical", "embedding_fallback_reason": "not_configured"}
        q = Counter(TOKEN.findall(query.lower()))
        tokenized = [set(TOKEN.findall(c.text.lower())) for c in self.chunks]
        scored = []
        for chunk in self.chunks:
            words = Counter(TOKEN.findall(chunk.text.lower()))
            overlap = 0.0
            for token, count in q.items():
                document_frequency = sum(token in tokens for tokens in tokenized)
                if document_frequency and words[token]:
                    overlap += min(count, words[token]) * (1.0 + math.log(1 + len(self.chunks) / document_frequency))
            phrase = sum(1.5 for token in q if token in words)
            scored.append(KnowledgeChunk(chunk.chunk_id, chunk.source, chunk.text, chunk.metadata, overlap + phrase))
        positive = sorted((x for x in scored if x.score > 0), key=lambda x: (-x.score, x.chunk_id))
        return positive[:top_k]


class Week4EmbeddingBackend:

    def __init__(self, artifact_dir, index_type="text", model_name="ViT-B-32", pretrained="laion2b_s34b_b79k"):
        from pathlib import Path
        import faiss
        import pandas as pd

        artifact_dir = Path(artifact_dir)
        index_path = artifact_dir / f"{index_type}.index"
        catalog_path = artifact_dir / "retrieval_catalog.parquet"
        if not index_path.is_file() or not catalog_path.is_file():
            raise FileNotFoundError("Week 4 index or retrieval catalog is missing")
        self.index = faiss.read_index(str(index_path))
        catalog = pd.read_parquet(catalog_path)
        id_column = next((x for x in ("item_id", "product_id", "id") if x in catalog.columns), None)
        if id_column is None or len(catalog) != self.index.ntotal:
            raise ValueError("Week 4 catalog/index alignment is invalid")
        self.product_ids = catalog[id_column].astype(str).tolist()
        self.model_name, self.pretrained = model_name, pretrained
        self._model = self._tokenizer = None

    def rank(self, query, chunks, top_k):
        import numpy as np
        import open_clip, torch

        if self._model is None:
            self._model, _, _ = open_clip.create_model_and_transforms(self.model_name, pretrained=self.pretrained)
            self._model.eval()
            self._tokenizer = open_clip.get_tokenizer(self.model_name)
        with torch.inference_mode():
            vector = self._model.encode_text(self._tokenizer([query])).float().cpu().numpy()
        vector = vector / np.clip(np.linalg.norm(vector, axis=1, keepdims=True), 1e-12, None)
        product_chunks = {c.metadata.get("product_id"): c for c in chunks if c.source == "product"}
        k = min(max(top_k * 2, top_k), self.index.ntotal)
        scores, positions = self.index.search(np.ascontiguousarray(vector, dtype=np.float32), k)
        ranked = []
        for score, position in zip(scores[0], positions[0]):
            if position < 0: continue
            chunk = product_chunks.get(self.product_ids[int(position)])
            if chunk:
                ranked.append(KnowledgeChunk(chunk.chunk_id, chunk.source, chunk.text, chunk.metadata, float(score)))
            if len(ranked) >= top_k: break
        return ranked

### Understand the customer query

This cell turns customer language into structured intent. Our active pipeline uses the rules parser. The parser recognizes requests such as “summer dinner,” “not too formal,” a specific category, a color, or a price limit.

In [50]:
from __future__ import annotations

import json, re
from typing import Protocol


SYSTEM_PROMPT = """You extract fashion customer intent. Use the retrieved context as knowledge, not as customer preferences.
Return only JSON matching the supplied schema. Preserve explicit requirements. Put desired attributes in style/colors and negative requirements in avoid. Use null or [] when unknown. Never invent a preference."""


class LLMBackend(Protocol):
    def extract(self, query: str, context: str, schema: dict) -> dict: ...


class OpenAIBackend:
    def __init__(self, api_key: str, model: str): self.api_key, self.model = api_key, model
    def extract(self, query, context, schema):
        from openai import OpenAI
        response = OpenAI(api_key=self.api_key).responses.create(
            model=self.model, input=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": f"Context:\n{context}\n\nCustomer query:\n{query}"}],
            text={"format": {"type": "json_schema", "name": "fashion_intent", "strict": True, "schema": schema}},
        )
        return json.loads(response.output_text)


class QueryParser:
    def __init__(self, llm: LLMBackend | None = None):
        self.llm = llm
        self.last_diagnostics = {"parser_mode": "rules_fallback", "llm_fallback_reason": "not_configured"}

    def parse(self, query: str, chunks=()) -> tuple[CustomerIntent, str]:
        context = "\n".join(f"[{x.source}] {x.text}" for x in chunks)
        if self.llm:
            try:
                intent = CustomerIntent.from_dict(self.llm.extract(query, context, INTENT_JSON_SCHEMA))
                self.last_diagnostics = {"parser_mode": "llm", "llm_fallback_reason": None}
                return intent, "llm"
            except Exception as exc:
                self.last_diagnostics = {"parser_mode": "rules_fallback", "llm_fallback_reason": type(exc).__name__}
        else:
            self.last_diagnostics = {"parser_mode": "rules_fallback", "llm_fallback_reason": "not_configured"}
        return self._rules(query), "rules_fallback"

    def _rules(self, query: str) -> CustomerIntent:
        text = " ".join(query.lower().split())
        avoid, spans = [], []
        pattern = re.compile(r"\b(?:not|avoid|no|without)\s+((?:too\s+)?[a-z]+(?:\s+[a-z]+){0,2}?)(?=\s+(?:and\s+(?:no|avoid|without|not)|but|for|with|under|below)\b|[,.;!?]|$)")
        for match in pattern.finditer(text):
            avoid.append(match.group(1).strip())
            spans.append(match.span())
        positive_text = text
        for start, end in reversed(spans): positive_text = positive_text[:start] + " " + positive_text[end:]

        styles = []
        if re.search(r"\bsmart[ -]casual\b", positive_text): styles.append("smart casual")
        for value in ("elegant", "casual", "formal", "minimal", "sporty", "bohemian", "classic"):
            if re.search(rf"\b{re.escape(value)}\b", positive_text):
                if value != "casual" or not re.search(r"\bsmart[ -]casual\b", positive_text): styles.append(value)
        if "elegant" in styles and any("formal" in x for x in avoid) and "smart casual" not in styles: styles.append("smart casual")
        if re.search(r"\bpolished\b", positive_text) and "classic" not in styles: styles.append("classic")
        season = next((x for x in ("spring", "summer", "autumn", "fall", "winter") if re.search(rf"\b{x}\b", positive_text)), None)
        categories = []
        mapping = {"tops": ("top", "shirt", "blouse"), "bottoms": ("bottom", "trouser", "pants", "skirt", "jeans"), "dresses": ("dress", "dresses"), "outerwear": ("jacket", "coat", "outerwear"), "shoes": ("shoe", "sneaker", "heel", "loafer", "boot"), "bags": ("bag", "handbag", "purse", "tote", "clutch"), "accessories": ("accessory", "accessories", "hat", "cap", "eyeglasses", "glasses", "sunglasses", "belt", "jewelry", "jewellery", "necklace", "earring")}
        for category, terms in mapping.items():
            if any(re.search(rf"\b{re.escape(term)}s?\b", positive_text) for term in terms): categories.append(category)
        outfit_words = ("outfit", "something", "look", "wear")
        if not categories and any(re.search(rf"\b{x}\b", positive_text) for x in outfit_words): categories = ["tops", "bottoms", "shoes", "bags", "accessories"]
        colors = [x for x in ("neutral", "light", "warm", "bright", "black", "white", "cream", "beige", "navy", "blue", "red", "green", "pink") if re.search(rf"\b{x}\b", positive_text)]
        if season == "summer" and "elegant" in styles and not colors: colors = ["neutral", "light", "warm"]
        maximum = None
        price = re.search(r"(?:under|below|up to|max(?:imum)?(?: of)?)\s*([$€£])?\s*(\d+(?:\.\d+)?)", text)
        currency = {"$": "USD", "€": "EUR", "£": "GBP"}.get(price.group(1)) if price else None
        if price: maximum = float(price.group(2))
        occasion = next((x for x in ("summer dinner", "dinner", "wedding", "work", "office", "party", "date", "travel", "everyday") if x in text), None)
        data = {"occasion": occasion, "style": styles, "avoid": avoid, "season": season, "categories_needed": categories, "color_palette": colors,
                "price_constraints": {"minimum": None, "maximum": maximum, "currency": currency} if maximum is not None else None}
        return CustomerIntent.from_dict(data)

### Filter and rank product candidates

This cell selects products from the requested categories, applies restrictions such as season, price, avoided terms, sneakers, and cotton, then ranks the remaining products by style, occasion, color, and related outfits. It also filters obvious category mistakes and clearly unsuitable office or dinner items. A price limit currently applies to each candidate product, not to the total price of a complete outfit.

In [58]:
from __future__ import annotations

import re
from collections import defaultdict

NEUTRAL = {"black", "white", "cream", "beige", "grey", "gray", "navy", "brown", "ivory", "tan"}
LIGHT = {"white", "cream", "beige", "ivory", "pastel", "light blue", "peach"}
WARM = {"cream", "beige", "ivory", "tan", "camel", "peach", "coral", "gold", "brown"}
BRIGHT = {"bright", "neon", "vivid", "red", "orange", "yellow", "lime", "hot pink", "fuchsia"}

STYLE_ALIASES = {"smart-casual": "smart casual"}


def _matches_color(color: str, requested: list[str]) -> bool:
    for item in requested:
        if item in color: return True
        if item == "neutral" and color in NEUTRAL: return True
        if item == "light" and color in LIGHT: return True
        if item == "warm" and color in WARM: return True
        if item == "bright" and color in BRIGHT: return True
    return False


def _norm(value):
    value = " ".join(str(value).lower().replace("-", " ").split())
    return STYLE_ALIASES.get(value, value)


def _taxonomy_hits(requested, actual):
    actual_set = {_norm(x) for x in actual}
    return [x for x in requested if _norm(x) in actual_set]


def _occasion_hits(requested, actual):
    aliases = {
        "office": {"office", "workwear", "work", "business"},
        "work": {"office", "workwear", "work", "business"},
        "summer dinner": {"summer dinner", "dinner", "evening out"},
        "dinner": {"dinner", "evening out"},
    }
    actual_set = {_norm(x) for x in actual}
    return [
        wanted for wanted in requested
        if actual_set & aliases.get(_norm(wanted), {_norm(wanted)})
    ]


def _excluded(product, avoided):
    haystack = " ".join([product["title"], product["description"], product["color"], *product["style"], *product["material"], *product["occasion"]])
    for raw in avoided:
        term = re.sub(r"^too\s+", "", _norm(raw))
        if term in {"bright", "bright color", "bright colors"} and product["color"] in BRIGHT: return True
        if term in {"high heel", "high heels"} and re.search(r"\b(?:high\s+)?heels?\b", haystack): return True
        if term and re.search(rf"\b{re.escape(term)}\b", haystack): return True
    return False

def _type_conflict(product):
    title = product["title"]
    category = product["category"]
    if category == "shoes":
        if title.startswith("a stylish pair of accessories:"):
            return True
        if re.search(
            r"\b(?:belts?|sunglasses?|eyeglasses?|glasses|eyewear|frames)\b",
            title,
        ):
            return True
    if category == "dresses":
        looks_like_separate = re.search(
            r"\b(?:shirt|blouse|top|skirt|shorts|pants|trousers)\b",
            title,
        )
        mentions_dress = re.search(r"\b(?:dress|gown)\b", title)
        if looks_like_separate and not mentions_dress:
            return True
    return False

def _query_requirements(query):
    text = _norm(query)
    return {
        "sneakers": bool(re.search(r"\b(?:sneakers?|trainers?)\b", text)),
        "cotton": bool(re.search(r"\bcotton\b", text)),
        "denim": bool(re.search(r"\bdenim\b", text)),
    }

def _occasion_type_conflict(product, intent):
    title = product["title"]
    obviously_sporty = re.search(
        r"\b(?:bras?|bralettes?|swimwear|bikini|workout|gym)\b", title
    )
    if intent.occasion in {"office", "work", "dinner", "summer dinner"}:
        if obviously_sporty:
            return True
    if intent.occasion in {"office", "work"}:
        if re.search(r"\b(?:sweatshirt|hoodie|plimsolls)\b", title):
            return True
    if intent.occasion in {"dinner", "summer dinner"}:
        if re.search(
            r"\b(?:sweatshirt|hoodie|ripped|distressed|hair elastics?)\b",
            title,
        ):
            return True
    return False

class CatalogCandidateRetriever:
    def __init__(self, products, outfits=None, min_relevance_score=2.0):
        self.products = products
        self.min_relevance_score = float(min_relevance_score)
        self.outfits = {str(x.get("outfit_id")): x for x in (outfits or []) if x.get("outfit_id")}
        self.product_outfits = defaultdict(set)
        for outfit_id, outfit in self.outfits.items():
            for product_id in outfit.get("product_ids", []): self.product_outfits[str(product_id)].add(outfit_id)

    def relevant_outfits(self, intent):
        relevant = set()
        for outfit_id, outfit in self.outfits.items():
            score = 0
            style = _style_values(outfit.get("style"))
            occasion = _listish(outfit.get("occasion"))
            colors = _listish(outfit.get("colors"))
            season = _listish(outfit.get("season"))
            if _taxonomy_hits(intent.style, style): score += 2
            if _occasion_hits([intent.occasion] if intent.occasion else [], occasion): score += 2
            if intent.season and _norm(intent.season) in {_norm(x) for x in season}: score += 1
            if intent.color_palette and any(_matches_color(color, intent.color_palette) for color in colors): score += 1
            if score >= 2: relevant.add(outfit_id)
        return relevant

    def retrieve(self, intent, per_category=5, min_relevance_score=None, relevant_outfit_ids=None, query=""):
        categories = intent.supported_categories()
        if not categories: return {}
        threshold = self.min_relevance_score if min_relevance_score is None else float(min_relevance_score)
        relevant_outfit_ids = set(relevant_outfit_ids or []) | self.relevant_outfits(intent)
        self.last_relevant_outfit_ids = sorted(relevant_outfit_ids)
        needs = _query_requirements(query)
        grouped = defaultdict(list)
        for p in self.products:
            if (p["category"] not in categories or _type_conflict(p) or _occasion_type_conflict(p, intent)): continue
            if intent.season and p["season"] and intent.season not in p["season"] and "all-season" not in p["season"]: continue
            pc = intent.price_constraints
            if pc and pc.minimum is not None and (p["price"] is None or p["price"] < pc.minimum): continue
            if pc and pc.maximum is not None and (p["price"] is None or p["price"] > pc.maximum): continue
            if _excluded(p, intent.avoid): continue
            if needs["sneakers"] and p["category"] == "shoes":
                if not re.search(r"\b(?:sneakers?|trainers?)\b", p["title"]): continue
            if needs["cotton"] and p["category"] in {"tops", "bottoms", "dresses", "outerwear"}:
                if "cotton" not in p["material"]: continue
            reasons, score = [], 0.0
            for label, requested, actual, weight in (("style", intent.style, p["style"], 3),):
                hits = _taxonomy_hits(requested, actual)
                if hits: score += weight * len(hits); reasons.append(f"{label}: {', '.join(hits)}")
            occasion_hits = _occasion_hits([intent.occasion] if intent.occasion else [], p["occasion"])
            if occasion_hits: score += 3 * len(occasion_hits); reasons.append(f"occasion: {', '.join(occasion_hits)}")
            if intent.color_palette and _matches_color(p["color"], intent.color_palette): score += 1; reasons.append(f"color: {p['color']}")
            if needs["sneakers"] and p["category"] == "shoes":
              score += 3
              reasons.append("type: sneakers")
            if needs["cotton"] and "cotton" in p["material"]:
              score += 3
              reasons.append("material: cotton")
            if needs["denim"] and (
              "denim" in p["material"] or re.search(r"\bdenim\b", p["title"])):
              score += 2
              reasons.append("denim")
            memberships = set(p["outfit_ids"]) | self.product_outfits.get(p["product_id"], set())
            relevant_memberships = memberships & relevant_outfit_ids
            if relevant_memberships: score += 0.5; reasons.append(f"relevant outfit: {sorted(relevant_memberships)[0]}")
            if score >= threshold:
                grouped[p["category"]].append({"product_id": p["product_id"], "title": p["title"], "score": round(score, 3), "price": p["price"], "reasons": reasons})
        return {cat: sorted(grouped.get(cat, []), key=lambda x: (-x["score"], x["product_id"]))[:per_category] for cat in categories}


def _listish(value):
    if value is None: return []
    if isinstance(value, list): return [_norm(x) for x in value]
    return [_norm(x) for x in re.split(r"[,|]", str(value)) if _norm(x)]


def _style_values(value):
    text = _norm(" ".join(value) if isinstance(value, list) else value or "")
    found = []
    if re.search(r"\bsmart casual\b", text):
        found.append("smart casual")
        text = re.sub(r"\bsmart casual\b", " ", text)
    for style in ("elegant", "casual", "formal", "minimal", "sporty", "bohemian", "classic"):
        if re.search(rf"\b{style}\b", text): found.append(style)
    return found

### Connect the pipeline

This cell joins the three main stages: retrieve relevant knowledge, parse the customer’s intent, and retrieve catalog products. `pipeline.run(query)` returns the structured intent, retrieved knowledge, validation issues, diagnostics, and ranked candidates.

In [59]:
from __future__ import annotations

class FashionRAGPipeline:
    def __init__(self, catalog, outfits=None, llm=None, embedding_backend=None, min_relevance_score=2.0):
        self.catalog = catalog
        self.outfits = outfits or []
        self.knowledge = KnowledgeRetriever(catalog, self.outfits, embedding_backend)
        self.parser = QueryParser(llm)
        self.ranker = CatalogCandidateRetriever(catalog, self.outfits, min_relevance_score)

    def run(self, query: str, knowledge_top_k=8, candidates_per_category=5, min_relevance_score=None):
        if not isinstance(query, str) or not query.strip(): raise ValueError("query must be non-empty")
        knowledge = self.knowledge.retrieve(query, knowledge_top_k)
        intent, parser_used = self.parser.parse(query, knowledge)
        issues = intent.validate()
        retrieved_outfit_ids = {str(x.metadata.get("outfit_id")) for x in knowledge if x.source == "outfit" and x.metadata.get("outfit_id")}
        relevant_outfit_ids = retrieved_outfit_ids & self.ranker.relevant_outfits(intent)
        candidates = self.ranker.retrieve(intent, candidates_per_category, min_relevance_score,relevant_outfit_ids, query=query)
        diagnostics = {**self.knowledge.last_diagnostics, **self.parser.last_diagnostics,
            "embedding_setup_fallback_reason": getattr(self, "embedding_setup_fallback_reason", None),
            "catalog_product_count": len(self.catalog),
            "relevant_outfit_ids": getattr(self.ranker, "last_relevant_outfit_ids", sorted(relevant_outfit_ids))}
        return {"query": query, "retrieved_knowledge": [x.to_dict() for x in knowledge], "intent": intent.to_dict(),
                "validation_issues": [x.__dict__ for x in issues], "parser": parser_used, "diagnostics": diagnostics, "candidates": candidates}

### Load the real catalog and existing outfits

This cell reads the refined catalog from Google Drive. Rows with the same `set_id` are grouped into existing outfits; products are identified by `item_id`. It then creates one normalized record per product and builds the rules-based pipeline. The segmentation manifest is loaded for project context but is not used to rank products in this cell.

In [60]:
import json
import re
import pandas as pd

MIN_RELEVANCE_SCORE = 2.0

raw_catalog = pd.read_parquet(CATALOG_PATH)
segmentation_manifest = pd.read_csv(SEGMENTATION_MANIFEST_PATH)

print(f'Raw catalog rows: {len(raw_catalog):,}')
print(f'Segmentation-manifest rows: {len(segmentation_manifest):,}')
print('Catalog columns:', raw_catalog.columns.tolist())

item_id_column = 'item_id'
set_id_column = 'set_id'

raw_catalog[item_id_column] = raw_catalog[item_id_column].astype(str).str.strip()
raw_catalog[set_id_column] = raw_catalog[set_id_column].astype(str).str.strip()
raw_catalog = raw_catalog[
    raw_catalog[item_id_column].ne('') & raw_catalog[set_id_column].ne('')
].copy()

raw_catalog = raw_catalog.drop_duplicates(
    subset=[set_id_column, item_id_column], keep='first'
).copy()

print('Refined categories:')
print(raw_catalog['refined_category'].value_counts(dropna=False))
print('Sample refined colors:')
print(raw_catalog['refined_color'].value_counts(dropna=False).head(20))


def unique_values(series):
    """Collect normalized comma/pipe-separated metadata without inventing values."""
    values = []
    for raw in series.dropna():
        for value in re.split(r'[,|]', str(raw)):
            value = ' '.join(value.strip().lower().split())
            if value and value not in values:
                values.append(value)
    return values


outfits = []
for set_id, group in raw_catalog.groupby(set_id_column, sort=False):
    product_ids = list(dict.fromkeys(group[item_id_column].astype(str)))
    if len(product_ids) < 2:
        continue

    outfit = {
        'outfit_id': str(set_id),
        'product_ids': product_ids,
    }
    outfit['style'] = unique_values(group['style'])
    outfit['occasion'] = unique_values(group['occasion'])
    outfit['season'] = unique_values(group['season'])
    outfit['colors'] = unique_values(group['refined_color'])
    categories = unique_values(group['refined_category'])
    outfit['name'] = f'Catalog outfit {set_id}'
    outfit['description'] = 'Existing outfit containing: ' + ', '.join(categories)
    outfits.append(outfit)

print(f'Created {len(outfits):,} existing outfits from {set_id_column}.')
if outfits:
    print('Sample outfit:')
    print(json.dumps(outfits[0], indent=2))

product_rows = raw_catalog.drop_duplicates(item_id_column, keep='first').copy()
catalog = [normalize_product(row) for row in product_rows.to_dict('records')]

pipeline = FashionRAGPipeline(
    catalog,
    outfits=outfits,
    min_relevance_score=MIN_RELEVANCE_SCORE,
)
CATALOG_USED = str(CATALOG_PATH)

print(f'Loaded {len(catalog):,} unique products from {CATALOG_USED}')
print('Sample normalized product:')
print(json.dumps(catalog[0], indent=2))


Raw catalog rows: 3,673
Segmentation-manifest rows: 3,673
Catalog columns: ['set_id', 'item_id', 'item_index_in_set', 'url_name', 'title', 'description', 'category', 'color', 'material', 'season', 'style', 'occasion', 'fit', 'formality', 'real_price', 'price_bucket', 'refined_category', 'refined_color']
Refined categories:
refined_category
accessories    867
shoes          789
bags           668
bottoms        454
tops           413
outerwear      256
dresses        226
Name: count, dtype: int64
Sample refined colors:
refined_color
black            617
blue             409
white            360
gold             329
beige            318
pink             313
red              237
brown            218
grey             191
green            186
silver           143
multi/neutral    114
yellow            90
purple            81
orange            67
Name: count, dtype: int64
Created 1,109 existing outfits from set_id.
Sample outfit:
{
  "outfit_id": "210750761",
  "product_ids": [
    "15424972

### Run one customer query

This cell runs the summer-dinner example and displays the parser used, structured intent, validation issues, and retrieved knowledge. Check that the intent reflects what the customer asked for; retrieved knowledge may still contain some less-relevant catalog text.

In [61]:
QUERY = 'I want something elegant but not too formal for a summer dinner.'
result = pipeline.run(QUERY)

print('PARSER:', result['parser'])
print('\nSTRUCTURED INTENT')
print(json.dumps(result['intent'], indent=2))
print('\nVALIDATION ISSUES')
print(json.dumps(result['validation_issues'], indent=2))
print('\nRETRIEVED KNOWLEDGE')
for chunk in result['retrieved_knowledge']:
    print(f"{chunk['score']:.3f} | {chunk['source']:<13} | {chunk['text']}")

PARSER: rules_fallback

STRUCTURED INTENT
{
  "occasion": "summer dinner",
  "style": [
    "elegant",
    "smart casual"
  ],
  "avoid": [
    "too formal"
  ],
  "season": "summer",
  "categories_needed": [
    "tops",
    "bottoms",
    "shoes",
    "bags",
    "accessories"
  ],
  "color_palette": [
    "neutral",
    "light",
    "warm"
  ],
  "price_constraints": null
}

VALIDATION ISSUES
[]

RETRIEVED KNOWLEDGE
41.212 | product       | pre-owned balmain pumps brand new! a classic black pump with beautiful detail! black patent leather balmain pumps with silver heel. absolutely stunning! sz 36 and too big for me. i wear a 5.5. has a 4.5" heel. made in france. doesn't include box but includes dust bags. not balmain for h&m shoes black leather all-season vintage everyday
33.327 | style         | Smart casual balances polished pieces with relaxed materials; it suits elegant events that are not too formal.
32.873 | product       | ugg women's classic tall sheepskin boots - chestnut th

### Inspect recommended candidates

This cell displays the products returned for the example query as a table, grouped by category and rank. It also verifies that every returned product ID exists in the loaded catalog. These are candidate items, not yet a complete, coordinated outfit.

In [62]:
import pandas as pd

candidate_rows = []
for category, products in result['candidates'].items():
    for rank, product in enumerate(products, start=1):
        candidate_rows.append({'category': category, 'rank': rank, **product})

candidate_table = pd.DataFrame(candidate_rows)
display(candidate_table)

catalog_ids = {product['product_id'] for product in catalog}
returned_ids = set(candidate_table['product_id']) if not candidate_table.empty else set()
assert returned_ids <= catalog_ids
print(f'Verified {len(returned_ids)} returned IDs against the selected catalog: {CATALOG_USED}')

,category,rank,product_id,title,score,price,reasons
0,tops,1,117635100,rails hunter plaid shirt in pine/white,4.5,40.47,"[occasion: summer dinner, color: multi/neutral..."
1,tops,2,131539979,a stylish pair of tops: isabel marant %c3%a9to...,4.5,25.20,"[occasion: summer dinner, color: white, releva..."
2,tops,3,146536948,rebecca taylor silk georgette drape top,4.5,30.20,"[occasion: summer dinner, color: white, releva..."
3,tops,4,149722016,a stylish pair of tops: music international fe...,4.5,20.73,"[occasion: summer dinner, color: white, releva..."
4,tops,5,152215249,baja east ribbed turtle neck crop top,4.5,29.66,"[occasion: summer dinner, color: white, releva..."
5,bottoms,1,133066819,sheinside women's white lace floral crochet sh...,4.5,60.01,"[occasion: summer dinner, color: white, releva..."
6,bottoms,2,143634795,wearall plus size skinny leg jeans,4.5,39.91,"[occasion: summer dinner, color: white, releva..."
7,bottoms,3,144423927,topshop tall leather-look pencil skirt,4.5,30.86,"[occasion: summer dinner, color: black, releva..."
8,bottoms,4,155010135,burberry classic skinny jeans,4.5,172.96,"[occasion: summer dinner, color: black, releva..."
9,bottoms,5,155432443,yoins suede high-rise button a-line mini skirt,4.5,58.67,"[occasion: summer dinner, color: black, releva..."


Verified 25 returned IDs against the selected catalog: /content/drive/MyDrive/catalog_refined.parquet


### Test important behaviors

This cell uses a small separate test catalog to check parsing, budgets, negative preferences, category mappings, product IDs, relevance thresholds, and fallback behavior. Passing these tests confirms those specific rules work; it does not guarantee that every recommendation from the full catalog is stylistically appropriate.

In [63]:
TEST_ROWS = [
    {'item_id':'T1','title':'Ivory Linen Top','description':'relaxed elegant','refined_category':'tops','refined_color':'ivory','material':'linen','season':'summer','style':'elegant,smart casual','occasion':'dinner','real_price':50},
    {'item_id':'T2','title':'Blue Denim Jeans','description':'casual everyday','refined_category':'bottoms','refined_color':'blue','material':'denim','season':'all-season','style':'casual','occasion':'everyday','real_price':70},
    {'item_id':'T3','title':'Formal Leather Shoes','description':'formal leather','refined_category':'shoes','refined_color':'black','material':'leather','season':'all-season','style':'formal','occasion':'dinner','real_price':120},
    {'item_id':'T4','title':'Gold Necklace','description':'elegant accent','refined_category':'accessories','refined_color':'gold','material':'metal','season':'all-season','style':'elegant','occasion':'dinner','real_price':30},
    {'item_id':'T5','title':'Beige Handbag','description':'elegant dinner bag','refined_category':'bags','refined_color':'beige','material':'leather','season':'all-season','style':'elegant','occasion':'dinner','real_price':45},
]
test_catalog = [normalize_product(row) for row in TEST_ROWS]
test_pipeline = FashionRAGPipeline(test_catalog)

class MalformedLLM:
    def extract(self, *args):
        raise ValueError('malformed output')

class FixedLLM:
    def __init__(self, value): self.value = value
    def extract(self, *args): return self.value

vague = test_pipeline.run('I need something nice')
assert vague['intent']['price_constraints'] is None
assert vague['intent']['season'] is None

budget = test_pipeline.run('an outfit under $60')
budget_items = [p for items in budget['candidates'].values() for p in items]
assert budget['intent']['price_constraints']['maximum'] == 60
assert all(p['price'] <= 60 for p in budget_items)

negative = test_pipeline.run('elegant summer dinner outfit but not too formal')
assert 'too formal' in negative['intent']['avoid']
assert 'T3' not in {p['product_id'] for items in negative['candidates'].values() for p in items}

missing, _ = QueryParser().parse('blue shoes')
assert missing.occasion is None and missing.season is None

conflict_data = {'occasion':None,'style':['formal'],'avoid':['formal'],'season':None,'categories_needed':['shoes'],'color_palette':[],'price_constraints':None}
conflict = FashionRAGPipeline(test_catalog, llm=FixedLLM(conflict_data)).run('formal but not formal shoes')
assert conflict['validation_issues'][0]['code'] == 'preference_conflict'

unsupported_data = {'occasion':None,'style':[],'avoid':[],'season':None,'categories_needed':['electronics'],'color_palette':[],'price_constraints':None}
unsupported = FashionRAGPipeline(test_catalog, llm=FixedLLM(unsupported_data)).run('electronics')
assert unsupported['validation_issues'][0]['code'] == 'unsupported_category'
assert unsupported['candidates'] == {}

_, parser_used = QueryParser(MalformedLLM()).parse('white summer shoes')
assert parser_used == 'rules_fallback'

assert KnowledgeRetriever([]).retrieve('zzzzunknown') == []

known = {p['product_id'] for p in test_catalog}
assert {p['product_id'] for items in negative['candidates'].values() for p in items} <= known

smart_only = [normalize_product({'item_id':'smart','title':'Smart top','refined_category':'tops','refined_color':'','style':'smart casual'})]
assert FashionRAGPipeline(smart_only).run('casual top')['candidates']['tops'] == []

dinner_only = [normalize_product({'item_id':'dinner','title':'Dinner top','refined_category':'tops','refined_color':'','occasion':'dinner','season':'summer'})]
assert FashionRAGPipeline(dinner_only).run('top for a summer dinner')['candidates']['tops'][0]['product_id'] == 'dinner'

assert QueryParser().parse('an elegant handbag')[0].categories_needed == ['bags']
assert QueryParser().parse('I want a hat')[0].categories_needed == ['accessories']
assert QueryParser().parse('I want eyeglasses')[0].categories_needed == ['accessories']
assert QueryParser().parse('summer dresses')[0].categories_needed == ['dresses']
assert test_pipeline.run('an elegant handbag')['candidates']['bags'][0]['product_id'] == 'T5'

no_inventory = normalize_product({'item_id':'no-inventory','title':'White sneaker','refined_category':'shoes','refined_color':'white','style':'minimal'})
assert FashionRAGPipeline([no_inventory]).run('minimal white sneakers')['candidates']['shoes'][0]['product_id'] == 'no-inventory'

strict = test_pipeline.run('minimal white sneakers', min_relevance_score=2.1)
assert all(p['score'] >= 2.1 for items in strict['candidates'].values() for p in items)

diagnostic = FashionRAGPipeline(test_catalog, llm=MalformedLLM()).run('white shoes')['diagnostics']
assert diagnostic['llm_fallback_reason'] == 'ValueError' and diagnostic['retrieval_mode'] == 'lexical'

multi, _ = QueryParser().parse('outfit, no bright colors, without leather, not too formal, avoid high heels')
assert multi.avoid == ['bright colors', 'leather', 'too formal', 'high heels']
assert 'bright' not in multi.color_palette and 'formal' not in multi.style

print('All Week 5 edge-case tests passed.')

All Week 5 edge-case tests passed.


### Review five end-to-end examples

This cell runs five different customer requests through the complete pipeline and prints their retrieved knowledge, structured intent, and product candidates. Use these examples to judge recommendation quality—not just whether the code runs. The output is a set of candidates for each category, not a finished outfit.

In [64]:
EXAMPLE_QUERIES = [
    'I want something elegant but not too formal for a summer dinner.',
    'A casual blue denim outfit for everyday wear under $80.',
    'Minimal white sneakers for summer.',
    'A warm cotton dress for a summer dinner under $100.',
    'Something polished for the office, no leather.',
]

example_results = []
for query in EXAMPLE_QUERIES:
    example = pipeline.run(query, knowledge_top_k=4, candidates_per_category=3)
    example_results.append(example)
    print('\n' + '=' * 100)
    print('CUSTOMER QUERY:', example['query'])
    print('\nRETRIEVED KNOWLEDGE:')
    print(json.dumps(example['retrieved_knowledge'], indent=2))
    print('\nSTRUCTURED INTENT:')
    print(json.dumps(example['intent'], indent=2))
    print('\nRETRIEVED PRODUCT CANDIDATES:')
    print(json.dumps(example['candidates'], indent=2))

print(f'\nCompleted {len(example_results)} end-to-end examples.')


CUSTOMER QUERY: I want something elegant but not too formal for a summer dinner.

RETRIEVED KNOWLEDGE:
[
  {
    "chunk_id": "product:156798799",
    "source": "product",
    "text": "pre-owned balmain pumps brand new! a classic black pump with beautiful detail! black patent leather balmain pumps with silver heel. absolutely stunning! sz 36 and too big for me. i wear a 5.5. has a 4.5\" heel. made in france. doesn't include box but includes dust bags. not balmain for h&m shoes black leather all-season vintage everyday",
    "score": 41.2115
  },
  {
    "chunk_id": "rule:0",
    "source": "style",
    "text": "Smart casual balances polished pieces with relaxed materials; it suits elegant events that are not too formal.",
    "score": 33.3272
  },
  {
    "chunk_id": "product:93939490",
    "source": "product",
    "text": "ugg women's classic tall sheepskin boots - chestnut these women's classic tall sheepskin boots by ugg australia have a classic design that uggs are famous for. with 